In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/emelierahn/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/emelierahn/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/emelierahn/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 711, in start
    self.io_loop.start()
  Fil

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [ ]:
# parameters
tau_e = 10 # time constants
tau_i = 20

a_ee = 10 # synaptic weights
a_ei = 12
a_ie = 8.5
a_ii = 3

theta_e = 2 # ???
theta_i = 3.5

sigma_e = 2 # ???
sigma_i = 5

g_e = 1
g_i = 1

K_0 = np.array([
    [a_ee, -a_ie],
    [a_ei, -a_ii]
])

In [ ]:
def K(sigma, x):
    return (1 / (sigma * math.sqrt(math.pi))) * math.exp(-(x**2) / (sigma**2))

def F(x): # firing rate
    return 1 / (1 + math.exp(-x))

def S(A, T, t): # stimulus
    """
    S(t) = AH(sin(2πt/T) - 0.8)
        A = amplitude
        T = period
        H(x) = 1 / (1 + e^(-50x))
          x = sin(2πt/T) - 0.8
    """
    x = math.sin(2 * math.pi * t / T) - 0.8
    return A * F(50*x)

In [ ]:
def wilson_cowan_eqs(V, A, T, t):
    V_e, V_i = V
    stim = S(A, T, t)
    
    input_e = a_ee * V_e - a_ie * V_i - theta_e + g_e * stim
    input_i = a_ei * V_e - a_ii * V_i - theta_i + g_i * stim
    
    dV_e = (-V_e + F(input_e)) / tau_e
    dV_i = (-V_i + F(input_i)) / tau_i

    return [dV_e, dV_i]

In [ ]:
A = 0.8 # amp
T = 55.0 # period
V_initial = [0.1, 0.1] # initial condition
number_of_cycles = 100

solution = solve_ivp(
    lambda t, V: wilson_cowan_eqs(V, A, T, t),
    (0, number_of_cycles * T),
    V_initial,
    dense_output = True,
    max_step = 0.1,
    rtol = 1e-8,
    atol = 1e-10
)

# Plot the final stimulus cycle
t_plot = np.linspace(
    number_of_cycles * T - T,
    number_of_cycles * T,
    1000
)

V_plot = solution.sol(t_plot)

plt.figure(figsize=(10, 5))
plt.plot(t_plot, V_plot[0], label = "Excitatory activity")
plt.plot(t_plot, V_plot[1], label = "Inhibitory activity")
plt.xlabel("Time (ms)")
plt.ylabel("Activity")
plt.title("Homogeneous Wilson-Cowan Model")
plt.legend()
plt.grid()
plt.show()

In [ ]:
t1 = np.linspace(
    (number_of_cycles - 2) * T,
    (number_of_cycles - 1) * T,
    500
)

t2 = t1 + T

cycle1 = solution.sol(t1)
cycle2 = solution.sol(t2)

periodicity_error = np.max(np.abs(cycle1 - cycle2))

print("Periodicity error:", periodicity_error)

In [ ]:
def prime_F(x): # derivative of F(x)
    # F'(x) = F(x)*(1 - F(x))
    return F(x) * (1 - F(x))

D = np.array([
    [1 / tau_e, 0],
    [0, 1 / tau_i]
])

def B(t, L, D): # time depen stability matrix
    """
    B_n(t) = -D + D*F'[K_0 * V(t) + S(t)] L_n
        t = time input
        V(t) = solution.sol(t)
        L_n = spatial connectivity eigenvalue matrix
        D = diagonal matrix of reciprocal time constants 
        S(t) = stimulus
    """
    V_t = solution.sol(t)
    stim = S(A, T, t)
    
    input_e = a_ee * V_t[0] - a_ie * V_t[1] - theta_e + g_e * stim
    input_i = a_ei * V_t[0] - a_ii * V_t[1] - theta_i + g_i * stim
    
    F_prime = np.array([
        [prime_F(input_e), 0],
        [0, prime_F(input_i)]
    ])
    
    return -D + D * F_prime @ L

def L(beta):
    return np.array([
        [
            a_ee * np.exp(-(beta**2 * sigma_e**2) / 4),
            -a_ie * np.exp(-(beta**2 * sigma_i**2) / 4)
        ],
        [
            a_ei * np.exp(-(beta**2 * sigma_e**2) / 4),
            -a_ii * np.exp(-(beta**2 * sigma_i**2) / 4)
        ]
    ])

def I(n):
    return np.identity(n)

def M(t, M_flat, L, D):
    # dM/dt = B(t)M(t), M(0) = I
    
    M = M_flat.reshape((2, 2))
    dM = B(t, L, D) @ M
    
    return dM.flatten()

def monodromy(beta):
    L_n = L(beta)
    
    t_start = solution.t[-1] - T
    t_end = solution.t[-1]
    
    M_0 = I(2).flatten()
    
    result = solve_ivp(
        M,
        (t_start, t_end),
        M_0,
        args = (L_n, D),
        rtol = 1e-8,
        atol = 1e-10,
        max_step = 0.1
    )

    M_T = result.y[:, -1].reshape((2, 2))

    return M_T

In [ ]:
beta = 0.5

M_T = monodromy(beta)

# Calculate eigenvalues
multipliers = np.linalg.eigvals(M_T)

print("Monodromy matrix:")
print(M_T)

print("\nFloquet multipliers:")
print(multipliers)

print("\nMagnitudes:")
print(np.abs(multipliers))

In [ ]:
if np.all(np.abs(multipliers) < 1):
    print("Stable")
else:
    print("Unstable")

In [ ]:
betas = np.linspace(0, 3, 100)

max_multipliers = []

for beta in betas:
    M_T = monodromy(beta)
    multipliers = np.linalg.eigvals(M_T)

    max_multipliers.append(np.max(np.abs(multipliers)))

plt.figure(figsize=(9, 5))
plt.plot(betas, max_multipliers)
plt.axhline(1, linestyle = "-", color = "red", label = "Stability boundary")
plt.xlabel("Spatial frequency (beta)")
plt.ylabel("Largest Floquet multiplier magnitude")
plt.title("Stability Across Spatial Frequencies")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# finding the eigenvalues

from scipy.optimize import root

A = 0
t = 0

equilibrium = root(
    lambda V: wilson_cowan_eqs(V, A, T, t), # V, A, T, t
    [0.1, 0.1]
)

V_star = equilibrium.x
print("Equilibrium:", V_star)

In [ ]:
input_e = a_ee * V_star[0] - a_ie * V_star[1] - theta_e
input_i = a_ei * V_star[0] - a_ii * V_star[1] - theta_i

F_prime = np.array([
    [prime_F(input_e), 0],
    [0, prime_F(input_i)]
])

J = -D + D @ F_prime @ K_0

print("Jacobian:")
print(J)

eigenvalues = np.linalg.eigvals(J)
print("Eigenvalues:", eigenvalues)

# Get the imaginary part of the eigenvalues
omega = np.abs(np.imag(eigenvalues))

# Convert from radians/ms to Hz
frequency = omega * 1000 / (2 * np.pi)

# Get the real part (decay rate)
decay_rate = np.real(eigenvalues)

print("Frequency (Hz):", frequency)
print("Decay rates:", decay_rate)

In [ ]:
def calculate_frequency(tau_e, tau_i):

    D_new = np.diag([
        1 / tau_e,
        1 / tau_i
    ])

    J_new = -D_new + D_new @ F_prime @ K_0

    eigenvalues = np.linalg.eigvals(J_new)

    # Check for complex eigenvalues
    if np.any(np.abs(np.imag(eigenvalues)) > 1e-10):
        omega = np.max(np.abs(np.imag(eigenvalues)))
        frequency = omega * 1000 / (2 * np.pi)
    else:
        frequency = 0

    return frequency, eigenvalues

In [ ]:
tau_e_values = np.linspace(5, 40, 100)
tau_i_fixed = 20

frequencies_e = []

for tau_e in tau_e_values:
    frequency, eigenvalues = calculate_frequency(tau_e, tau_i_fixed)
    frequencies_e.append(frequency)
    
tau_i_values = np.linspace(5, 40, 100)
tau_e_fixed = 10

frequencies_i = []

for tau_i in tau_i_values:
    frequency, eigenvalues = calculate_frequency(tau_e_fixed, tau_i)
    frequencies_i.append(frequency)

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    tau_e_values,
    frequencies_e,
    label="Varying tau_e (tau_i = 20 ms)",
    linewidth=2
)

plt.plot(
    tau_i_values,
    frequencies_i,
    label="Varying tau_i (tau_e = 10 ms)",
    linewidth=2
)

plt.axhline(
    13,
    linestyle="--",
    label="Paper: 13 Hz"
)

plt.xlabel("Time constant (ms)")
plt.ylabel("Oscillation frequency (Hz)")
plt.title("Effect of Time Constants on Oscillation Frequency")

plt.legend()
plt.grid(True)
plt.show()